# Day 3：Self-Instruct 数据生成 — 手写自动化指令数据管线

> **目标**：精读 Self-Instruct 论文的核心思想，理解从种子任务出发，利用 LLM 自动化生成大规模高质量指令数据的完整流程；手写一个 Self-Instruct 数据生成管线，涵盖指令生成、输入构造、输出生成、质量过滤和多样性控制；深入分析数据质量与模型效果的关系。

---

## 总体流程

```
Part 1: Self-Instruct 论文核心思想
  种子任务 → 指令生成 → 分类判断 → 输入/输出生成 → 过滤

Part 2: 手写 Self-Instruct 管线
  种子任务设计 → LLM 调用 → 指令生成 → 质量过滤 → 多样性控制

Part 3: 数据质量分析
  统计分析 → 质量评估 → 与 Alpaca 数据对比

Part 4: 扩展——Evol-Instruct 思路实现
  复杂度递增 → 指令进化 → 对比原始 Self-Instruct
```

In [1]:
import json
import random
import re
import time
from collections import Counter
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np

random.seed(42)
np.random.seed(42)

## Part 1：Self-Instruct 论文核心思想

### 1.1 问题定义

**论文**：*Self-Instruct: Aligning Language Models with Self-Generated Instructions* (Wang et al., 2023)

**核心问题**：高质量的指令微调数据需要大量人工标注，成本极高。能否让 LLM 自己生成训练数据？

```
传统路径（昂贵）:
  雇佣标注员 → 编写指令 → 审核质量 → 获得数据
  成本: 数万~数十万美元
  周期: 数周~数月

Self-Instruct 路径（廉价）:
  175 条种子任务 → LLM 自动生成 → 自动过滤 → 获得数据
  成本: ~$500（API 调用费）
  周期: 数小时~1天
```

### 1.2 核心流程（四阶段）

```
Stage 1: 指令生成（Instruction Generation）
  从种子任务池中采样 → 作为 few-shot 示例 → 让 LLM 生成新指令
       │
       ▼
Stage 2: 分类判断（Classification Identification）
  判断新指令是否为分类任务（影响后续 input/output 生成策略）
       │
       ▼
Stage 3: 实例生成（Instance Generation）
  分类任务 → output-first 策略（先生成类别，再构造输入）
  非分类任务 → input-first 策略（先生成输入，再生成输出）
       │
       ▼
Stage 4: 质量过滤（Filtering）
  去重（ROUGE-L 相似度）→ 格式检查 → 长度过滤
```

### 1.3 关键设计决策

| 设计点 | Self-Instruct 的选择 | 理由 |
|--------|---------------------|------|
| 种子任务数量 | 175 条 | 足够覆盖主要任务类型 |
| 每轮生成数量 | 每个 prompt 生成 1 条指令 | 保证多样性 |
| few-shot 采样 | 6 条人工 + 2 条已生成 | 平衡质量与多样性 |
| 分类/非分类区分 | 用 LLM 判断 | 不同任务类型需要不同生成策略 |
| 去重阈值 | ROUGE-L > 0.7 视为重复 | 经验值 |

### 1.4 Self-Instruct vs Alpaca 的简化版本

```
原始 Self-Instruct:
  ✅ 分类判断步骤
  ✅ output-first / input-first 两种策略
  ✅ 每个 prompt 生成 1 条指令
  ✅ 多轮迭代（生成的指令加入池子）
  ✅ 精细的 ROUGE-L 去重

Alpaca 简化版:
  ❌ 省略分类判断
  ❌ 统一生成策略（不区分分类/非分类）
  ✅ 每个 prompt 生成 20 条指令（更高效）
  ❌ 省略多轮迭代
  ⚠️ 简单去重（非 ROUGE-L）

→ Alpaca 版本更简单、更快，但数据多样性和质量略有下降
```

## Part 2：手写 Self-Instruct 数据管线

### 2.1 种子任务设计

种子任务是整个管线的基础。精心设计的种子任务直接决定了生成数据的质量和多样性。

In [ ]:
SEED_TASKS = [
    # ====== 文本生成 ======
    {
        "instruction": "Write a short poem about the beauty of autumn.",
        "input": "",
        "output": "Golden leaves descend in gentle flight,\nPainting paths in amber light.\nCrisp air whispers through the trees,\nCarrying autumn's fragrant breeze.",
        "category": "generation"
    },
    {
        "instruction": "Write a professional email declining a meeting invitation.",
        "input": "Meeting: Q3 Budget Review, Time: Monday 2pm",
        "output": "Subject: Re: Q3 Budget Review - Unable to Attend\n\nDear Team,\n\nThank you for the invitation to the Q3 Budget Review on Monday at 2pm. Unfortunately, I have a prior commitment at that time. Could we reschedule, or would it be possible to share the meeting notes afterward?\n\nBest regards",
        "category": "generation"
    },

    # ====== 问答 ======
    {
        "instruction": "What is the difference between supervised and unsupervised learning?",
        "input": "",
        "output": "Supervised learning uses labeled data where the model learns to map inputs to known outputs (e.g., classification, regression). Unsupervised learning works with unlabeled data, discovering hidden patterns or structures (e.g., clustering, dimensionality reduction). The key difference is whether training data includes target labels.",
        "category": "qa"
    },
    {
        "instruction": "Explain the concept in simple terms.",
        "input": "Gradient descent is an optimization algorithm used to minimize some function by iteratively moving in the direction of steepest descent as defined by the negative of the gradient.",
        "output": "Imagine you're blindfolded on a hilly landscape and want to reach the lowest valley. You feel the ground slope beneath your feet and take a step downhill. You repeat this process, always stepping in the steepest downhill direction. That's gradient descent — it helps find the lowest point (best solution) by repeatedly moving in the direction that decreases the most.",
        "category": "qa"
    },

    # ====== 分类 ======
    {
        "instruction": "Classify the sentiment of the following review as positive, negative, or neutral.",
        "input": "The food was decent but the service was incredibly slow. We waited 45 minutes for our appetizers.",
        "output": "Negative. While the reviewer acknowledges the food quality was acceptable, the overall experience was negative due to extremely slow service.",
        "category": "classification"
    },
    {
        "instruction": "Determine whether the following statement is a fact or an opinion.",
        "input": "Python is the best programming language for data science.",
        "output": "Opinion. While Python is widely used and popular for data science, calling it 'the best' is a subjective judgment. Other languages like R, Julia, and Scala also have strong data science ecosystems.",
        "category": "classification"
    },

    # ====== 改写与摘要 ======
    {
        "instruction": "Summarize the following paragraph in one sentence.",
        "input": "The transformer architecture, introduced in 2017, revolutionized natural language processing by replacing recurrent networks with self-attention mechanisms. This allowed for much better parallelization during training and enabled models to capture long-range dependencies more effectively. The architecture became the foundation for models like BERT, GPT, and T5.",
        "output": "The transformer architecture (2017) replaced recurrent networks with self-attention, enabling better parallelization and long-range dependency modeling, and became the foundation for BERT, GPT, and T5.",
        "category": "rewriting"
    },
    {
        "instruction": "Rewrite the following sentence to make it more formal.",
        "input": "The project's gonna be done by next week, no worries.",
        "output": "The project is expected to be completed by next week. Please rest assured that we are on track.",
        "category": "rewriting"
    },

    # ====== 代码 ======
    {
        "instruction": "Write a Python function that checks if a string is a palindrome.",
        "input": "",
        "output": "def is_palindrome(s: str) -> bool:\n    s = s.lower().replace(' ', '')\n    return s == s[::-1]\n\n# Examples:\n# is_palindrome('racecar')  -> True\n# is_palindrome('hello')    -> False\n# is_palindrome('A man a plan a canal Panama') -> True",
        "category": "code"
    },
    {
        "instruction": "Explain what the following code does.",
        "input": "sorted(set(words), key=lambda w: (-len(w), w))",
        "output": "This code: (1) removes duplicate words using set(), (2) sorts the unique words first by length in descending order (-len(w)), and (3) for words of equal length, sorts alphabetically (w). The result is a list of unique words ordered from longest to shortest, with alphabetical ordering as a tiebreaker.",
        "category": "code"
    },

    # ====== 推理 ======
    {
        "instruction": "Solve the following math problem step by step.",
        "input": "If a train travels at 60 km/h for 2.5 hours, then at 80 km/h for 1.5 hours, what is the total distance traveled?",
        "output": "Step 1: Distance in first segment = speed × time = 60 × 2.5 = 150 km\nStep 2: Distance in second segment = speed × time = 80 × 1.5 = 120 km\nStep 3: Total distance = 150 + 120 = 270 km\n\nThe train traveled a total of 270 km.",
        "category": "reasoning"
    },
    {
        "instruction": "What logical fallacy is present in the following argument?",
        "input": "Everyone I know likes this movie, so it must be the best movie ever made.",
        "output": "This is a hasty generalization fallacy. The speaker is drawing a broad conclusion ('best movie ever') from a limited sample (only people they personally know). Their social circle is not representative of all moviegoers, and personal popularity doesn't equate to objective quality.",
        "category": "reasoning"
    },

    # ====== 信息提取 ======
    {
        "instruction": "Extract all the dates mentioned in the text.",
        "input": "The project started on January 15, 2023. The first milestone was completed on March 3, 2023. The final deadline is December 31, 2023.",
        "output": "1. January 15, 2023 (project start)\n2. March 3, 2023 (first milestone)\n3. December 31, 2023 (final deadline)",
        "category": "extraction"
    },

    # ====== 翻译 ======
    {
        "instruction": "Translate the following sentence into Chinese.",
        "input": "Artificial intelligence is transforming every aspect of our daily lives.",
        "output": "人工智能正在改变我们日常生活的方方面面。",
        "category": "translation"
    },

    # ====== 创意 ======
    {
        "instruction": "Come up with 3 creative names for a coffee shop that specializes in books and coffee.",
        "input": "",
        "output": "1. Brewed Pages — Combining the idea of brewing coffee with turning pages\n2. The Caffeinated Chapter — Suggesting each visit is a new chapter fueled by caffeine\n3. Bean & Bookmark — A simple, memorable pairing of the two core offerings",
        "category": "generation"
    }
]

print(f"种子任务数量: {len(SEED_TASKS)}")
categories = Counter(t['category'] for t in SEED_TASKS)
print(f"类别分布: {dict(categories)}")

### 2.2 指令生成模块

Self-Instruct 的核心是让 LLM 基于少量 few-shot 示例生成新指令。

由于我们可能没有 OpenAI API 访问权限，这里实现两个版本：
- **模拟版本**：用规则模拟 LLM 生成行为（用于理解流程）
- **API 版本**：调用真实 LLM API（如有条件）

In [ ]:
def build_instruction_generation_prompt(
    seed_tasks: List[Dict],
    generated_tasks: List[Dict],
    num_seed_examples: int = 6,
    num_generated_examples: int = 2
) -> str:
    """
    构建指令生成的 prompt。
    遵循 Self-Instruct 论文：从种子任务中采样 6 条 + 已生成任务中采样 2 条
    作为 few-shot 示例。
    """
    sampled_seeds = random.sample(
        seed_tasks, min(num_seed_examples, len(seed_tasks))
    )
    sampled_generated = []
    if generated_tasks:
        sampled_generated = random.sample(
            generated_tasks,
            min(num_generated_examples, len(generated_tasks))
        )

    all_examples = sampled_seeds + sampled_generated
    random.shuffle(all_examples)

    prompt = (
        "Come up with a series of tasks:\n\n"
    )

    for i, task in enumerate(all_examples, 1):
        prompt += f"Task {i}: {task['instruction']}\n"

    prompt += f"Task {len(all_examples) + 1}:"

    return prompt


prompt = build_instruction_generation_prompt(SEED_TASKS, [])
print("===== 生成的 Prompt =====")
print(prompt)

In [ ]:
def build_instance_generation_prompt(
    instruction: str,
    seed_tasks: List[Dict],
    is_classification: bool = False
) -> str:
    """
    构建实例生成（input + output）的 prompt。

    对于分类任务，采用 output-first 策略（先确定类别再构造输入）。
    对于非分类任务，采用 input-first 策略（先构造输入再生成输出）。
    """
    similar_tasks = random.sample(seed_tasks, min(3, len(seed_tasks)))

    if is_classification:
        prompt = (
            "Given the classification task, generate an input-output pair. "
            "First decide the output class, then create a realistic input.\n\n"
        )
        for i, task in enumerate(similar_tasks, 1):
            prompt += f"Example {i}:\n"
            prompt += f"Instruction: {task['instruction']}\n"
            prompt += f"Output: {task['output']}\n"
            if task['input']:
                prompt += f"Input: {task['input']}\n"
            prompt += "\n"

        prompt += f"Now generate for:\nInstruction: {instruction}\nOutput:"
    else:
        prompt = (
            "Given the following task instruction, generate a realistic input "
            "(if needed) and an appropriate output.\n\n"
        )
        for i, task in enumerate(similar_tasks, 1):
            prompt += f"Example {i}:\n"
            prompt += f"Instruction: {task['instruction']}\n"
            if task['input']:
                prompt += f"Input: {task['input']}\n"
            prompt += f"Output: {task['output']}\n\n"

        prompt += f"Now generate for:\nInstruction: {instruction}\n"

    return prompt


prompt_cls = build_instance_generation_prompt(
    "Classify whether the email is spam or not.",
    SEED_TASKS,
    is_classification=True
)
print("===== 分类任务 Instance 生成 Prompt =====")
print(prompt_cls[:500])
print("\n...\n")

prompt_gen = build_instance_generation_prompt(
    "Write a function to calculate fibonacci numbers.",
    SEED_TASKS,
    is_classification=False
)
print("===== 非分类任务 Instance 生成 Prompt =====")
print(prompt_gen[:500])

### 2.3 分类判断模块

Self-Instruct 论文中，分类判断用于决定实例生成策略：
- **分类任务**：output 是有限类别之一（如正面/负面、是/否），采用 output-first 策略
- **非分类任务**：output 是开放式文本（如写文章、回答问题），采用 input-first 策略

In [ ]:
CLASSIFICATION_KEYWORDS = [
    "classify", "categorize", "determine whether", "is it",
    "label", "predict if", "identify whether", "sentiment",
    "positive or negative", "true or false", "yes or no",
    "fact or opinion", "spam or", "toxic or"
]


def is_classification_task(instruction: str) -> bool:
    """
    判断指令是否为分类任务。
    简化实现：基于关键词匹配（论文中使用 LLM 判断）。
    """
    instruction_lower = instruction.lower()
    return any(kw in instruction_lower for kw in CLASSIFICATION_KEYWORDS)


test_instructions = [
    "Classify the sentiment of the review.",
    "Write a poem about the ocean.",
    "Determine whether the statement is true or false.",
    "Translate the sentence to Spanish.",
    "Is it a fact or an opinion?",
    "Explain quantum computing in simple terms.",
]

for inst in test_instructions:
    result = is_classification_task(inst)
    print(f"{'[分类]' if result else '[非分类]'} {inst}")

### 2.4 质量过滤模块

Self-Instruct 的质量过滤包括多个维度：
1. **去重**：ROUGE-L 相似度检查，避免生成重复指令
2. **格式检查**：确保 instruction/input/output 格式正确
3. **长度过滤**：过滤过短或过长的数据
4. **内容过滤**：排除不适当内容

In [ ]:
def compute_rouge_l(reference: str, candidate: str) -> float:
    """
    计算 ROUGE-L (基于最长公共子序列 LCS) 的 F1 分数。

    ROUGE-L 衡量两段文本之间的相似度，值域 [0, 1]。
    在 Self-Instruct 中用于检测生成指令是否与已有指令重复。
    """
    ref_words = reference.lower().split()
    cand_words = candidate.lower().split()

    if not ref_words or not cand_words:
        return 0.0

    m, n = len(ref_words), len(cand_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_words[i - 1] == cand_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs_len = dp[m][n]
    precision = lcs_len / n if n > 0 else 0
    recall = lcs_len / m if m > 0 else 0

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# 验证 ROUGE-L 实现
print("ROUGE-L 测试:")
print(f"  完全相同: {compute_rouge_l('the cat sat on the mat', 'the cat sat on the mat'):.3f}")
print(f"  部分重叠: {compute_rouge_l('the cat sat on the mat', 'the cat is on the mat'):.3f}")
print(f"  完全不同: {compute_rouge_l('hello world', 'good morning'):.3f}")
print(f"  相似指令: {compute_rouge_l('classify the sentiment', 'classify the sentiment of the review'):.3f}")

In [ ]:
@dataclass
class FilterConfig:
    """质量过滤配置。"""
    rouge_l_threshold: float = 0.7
    min_instruction_len: int = 10
    max_instruction_len: int = 500
    min_output_len: int = 5
    max_output_len: int = 2000
    blacklist_words: List[str] = field(default_factory=lambda: [
        "image", "picture", "graph", "plot", "figure",
        "video", "audio", "voice", "sing", "draw"
    ])


class QualityFilter:
    """Self-Instruct 质量过滤器。"""

    def __init__(self, config: FilterConfig = None):
        self.config = config or FilterConfig()
        self.existing_instructions: List[str] = []
        self.filter_stats = Counter()

    def add_existing(self, instructions: List[str]):
        self.existing_instructions.extend(instructions)

    def check_duplicate(self, instruction: str) -> bool:
        """ROUGE-L 去重：与已有指令相似度超过阈值则判为重复。"""
        for existing in self.existing_instructions:
            score = compute_rouge_l(existing, instruction)
            if score > self.config.rouge_l_threshold:
                return True
        return False

    def check_format(self, task: Dict) -> bool:
        """格式检查：确保必要字段存在且非空。"""
        if not task.get("instruction", "").strip():
            return False
        if not task.get("output", "").strip():
            return False
        return True

    def check_length(self, task: Dict) -> bool:
        """长度过滤：过滤过短或过长的指令和输出。"""
        inst_len = len(task["instruction"])
        out_len = len(task["output"])
        return (
            self.config.min_instruction_len <= inst_len <= self.config.max_instruction_len
            and self.config.min_output_len <= out_len <= self.config.max_output_len
        )

    def check_content(self, task: Dict) -> bool:
        """内容过滤：排除 LLM 无法完成的任务（如生成图片、音频）。"""
        text = (task["instruction"] + " " + task.get("input", "")).lower()
        return not any(word in text for word in self.config.blacklist_words)

    def filter(self, task: Dict) -> Tuple[bool, str]:
        """综合过滤，返回 (是否通过, 原因)。"""
        if not self.check_format(task):
            self.filter_stats["format"] += 1
            return False, "format_error"

        if not self.check_length(task):
            self.filter_stats["length"] += 1
            return False, "length_violation"

        if not self.check_content(task):
            self.filter_stats["content"] += 1
            return False, "blacklisted_content"

        if self.check_duplicate(task["instruction"]):
            self.filter_stats["duplicate"] += 1
            return False, "duplicate"

        self.filter_stats["passed"] += 1
        return True, "passed"


# 测试过滤器
qf = QualityFilter()
qf.add_existing([t["instruction"] for t in SEED_TASKS])

test_tasks = [
    {"instruction": "Write a short poem about the beauty of autumn.",
     "input": "", "output": "Leaves fall gently."},
    {"instruction": "Explain the theory of relativity in simple terms.",
     "input": "", "output": "Einstein's theory states that space and time are intertwined."},
    {"instruction": "Draw a picture of a cat.",
     "input": "", "output": "Here is a cat."},
    {"instruction": "Hi",
     "input": "", "output": "Hello!"},
]

for task in test_tasks:
    passed, reason = qf.filter(task)
    status = "✅ 通过" if passed else f"❌ 拒绝 ({reason})"
    print(f"{status}: {task['instruction'][:60]}")

print(f"\n过滤统计: {dict(qf.filter_stats)}")

### 2.5 多样性控制模块

多样性是 Self-Instruct 数据质量的关键。我们从三个维度衡量和控制多样性：
1. **动词多样性**：指令开头的动词应尽可能不同
2. **任务类型多样性**：覆盖多种任务类型
3. **长度多样性**：指令和输出的长度应有适当变化

In [ ]:
class DiversityController:
    """多样性控制器：监控并引导数据的多样性。"""

    def __init__(self, max_verb_ratio: float = 0.1):
        self.verb_counter = Counter()
        self.category_counter = Counter()
        self.total_count = 0
        self.max_verb_ratio = max_verb_ratio

    def extract_root_verb(self, instruction: str) -> str:
        """提取指令的首个动词（简化版：取第一个单词）。"""
        words = instruction.strip().split()
        if words:
            return words[0].lower().rstrip('.,!?:;')
        return "unknown"

    def check_diversity(self, instruction: str) -> bool:
        """
        检查新指令是否满足多样性要求。
        如果某个动词出现比例超过阈值，拒绝该指令。
        """
        verb = self.extract_root_verb(instruction)
        if self.total_count > 0:
            current_ratio = self.verb_counter.get(verb, 0) / self.total_count
            if current_ratio >= self.max_verb_ratio:
                return False
        return True

    def add(self, instruction: str, category: str = "unknown"):
        verb = self.extract_root_verb(instruction)
        self.verb_counter[verb] += 1
        self.category_counter[category] += 1
        self.total_count += 1

    def report(self) -> Dict:
        """生成多样性报告。"""
        return {
            "total_tasks": self.total_count,
            "unique_verbs": len(self.verb_counter),
            "top_10_verbs": self.verb_counter.most_common(10),
            "category_distribution": dict(self.category_counter),
        }


# 测试多样性控制器
dc = DiversityController(max_verb_ratio=0.15)
for task in SEED_TASKS:
    dc.add(task["instruction"], task.get("category", "unknown"))

report = dc.report()
print(f"总任务数: {report['total_tasks']}")
print(f"唯一动词数: {report['unique_verbs']}")
print(f"Top 10 动词: {report['top_10_verbs']}")
print(f"类别分布: {report['category_distribution']}")

### 2.6 完整 Self-Instruct 管线

将以上模块整合为完整的数据生成管线。由于没有真实 LLM API，我们用模拟生成来演示完整流程。

In [ ]:
# 模拟 LLM 生成（用模板变换替代真实 API 调用）
INSTRUCTION_TEMPLATES = [
    "Write a {adj} story about {topic}.",
    "Explain the concept of {concept} to a {audience}.",
    "List {n} reasons why {topic} is important.",
    "Compare and contrast {thing1} and {thing2}.",
    "Summarize the following {content_type} in {n} sentences.",
    "Create a {format} for {purpose}.",
    "Identify the main {element} in the text.",
    "Rewrite the following text to be more {style}.",
    "What are the advantages and disadvantages of {topic}?",
    "Provide step-by-step instructions for {task}.",
    "Classify the following text as {class1} or {class2}.",
    "Generate {n} examples of {type}.",
    "Correct the grammar errors in the following sentence.",
    "Translate the following text from {lang1} to {lang2}.",
    "Write a Python function that {functionality}.",
    "Analyze the {aspect} of the given {content}.",
    "Design a {thing} that {requirement}.",
    "Evaluate whether the argument is {quality}.",
    "Describe the process of {process} in detail.",
    "Suggest {n} improvements for {thing}.",
]

FILL_VALUES = {
    "adj": ["short", "creative", "suspenseful", "humorous", "inspiring"],
    "topic": ["climate change", "space exploration", "artificial intelligence",
              "remote work", "renewable energy", "education reform"],
    "concept": ["machine learning", "blockchain", "quantum computing",
                "neural networks", "cloud computing"],
    "audience": ["5-year-old", "high school student", "non-technical manager",
                 "college freshman"],
    "n": ["3", "5", "7", "10"],
    "thing1": ["Python", "democracy", "online learning", "electric cars"],
    "thing2": ["Java", "authoritarianism", "in-person learning", "gas cars"],
    "content_type": ["article", "research paper", "book chapter", "news report"],
    "format": ["template", "checklist", "outline", "schedule"],
    "purpose": ["a job interview", "a research proposal", "a weekly plan",
                "a project kickoff"],
    "element": ["argument", "theme", "tone", "bias"],
    "style": ["formal", "concise", "persuasive", "friendly"],
    "class1": ["positive", "factual", "formal", "urgent"],
    "class2": ["negative", "opinion-based", "informal", "non-urgent"],
    "type": ["metaphors", "analogies", "compound sentences", "rhetorical questions"],
    "lang1": ["English", "Chinese", "French"],
    "lang2": ["Spanish", "English", "German"],
    "functionality": ["sorts a dictionary by values", "finds prime numbers",
                       "merges two sorted lists", "validates email addresses"],
    "aspect": ["tone", "structure", "main argument", "writing style"],
    "content": ["essay", "speech", "poem", "email"],
    "thing": ["study plan", "workout routine", "meal plan", "reading list"],
    "requirement": ["fits a busy schedule", "costs under $50",
                     "can be done at home", "takes less than 30 minutes"],
    "quality": ["logically sound", "well-supported", "biased", "convincing"],
    "process": ["photosynthesis", "machine learning training",
                "making sourdough bread", "debugging code"],
}


def simulate_llm_instruction_generation() -> str:
    """模拟 LLM 生成一条新指令。"""
    template = random.choice(INSTRUCTION_TEMPLATES)
    placeholders = re.findall(r'\{(\w+)\}', template)
    for ph in placeholders:
        if ph in FILL_VALUES:
            template = template.replace(
                f"{{{ph}}}", random.choice(FILL_VALUES[ph]), 1
            )
    return template


def simulate_llm_output_generation(instruction: str, input_text: str = "") -> str:
    """模拟 LLM 生成 output。"""
    outputs = [
        f"Here is a response to the instruction: '{instruction[:50]}...' "
        f"This is a detailed and helpful answer that addresses the key points "
        f"of the instruction. The response provides relevant information and "
        f"examples to support the main ideas.",

        f"To address this task, we need to consider several important factors. "
        f"First, let's understand the context. Then, we can provide a "
        f"comprehensive answer that covers all aspects of the question.",

        f"This is an excellent question. The answer involves understanding "
        f"multiple perspectives. Here is a balanced and thorough response "
        f"that considers both sides of the issue.",
    ]
    return random.choice(outputs)


# 测试
for i in range(5):
    inst = simulate_llm_instruction_generation()
    print(f"  Generated: {inst}")

In [ ]:
class SelfInstructPipeline:
    """
    Self-Instruct 完整管线。

    实现论文中的四阶段流程：
    1. 指令生成
    2. 分类判断
    3. 实例生成
    4. 质量过滤
    """

    def __init__(
        self,
        seed_tasks: List[Dict],
        target_count: int = 100,
        filter_config: FilterConfig = None,
        max_verb_ratio: float = 0.15,
    ):
        self.seed_tasks = seed_tasks
        self.target_count = target_count
        self.generated_tasks: List[Dict] = []
        self.quality_filter = QualityFilter(filter_config or FilterConfig())
        self.diversity_ctrl = DiversityController(max_verb_ratio)

        # 初始化：将种子任务加入过滤器和多样性控制器
        self.quality_filter.add_existing(
            [t["instruction"] for t in seed_tasks]
        )
        for t in seed_tasks:
            self.diversity_ctrl.add(
                t["instruction"], t.get("category", "unknown")
            )

        self.stats = {
            "total_attempts": 0,
            "passed": 0,
            "filtered": Counter(),
        }

    def generate_one(self) -> Optional[Dict]:
        """
        生成一条指令数据（模拟版）。
        完整流程：生成指令 → 分类判断 → 生成实例 → 质量过滤。
        """
        self.stats["total_attempts"] += 1

        # Stage 1: 指令生成
        instruction = simulate_llm_instruction_generation()

        # Stage 2: 分类判断
        is_cls = is_classification_task(instruction)

        # Stage 3: 实例生成
        has_input = random.random() < 0.4
        input_text = ""
        if has_input:
            input_text = f"Sample input text for the task: {instruction[:30]}..."
        output_text = simulate_llm_output_generation(instruction, input_text)

        task = {
            "instruction": instruction,
            "input": input_text,
            "output": output_text,
            "is_classification": is_cls,
        }

        # Stage 4: 质量过滤
        passed, reason = self.quality_filter.filter(task)
        if not passed:
            self.stats["filtered"][reason] += 1
            return None

        # 多样性检查
        if not self.diversity_ctrl.check_diversity(instruction):
            self.stats["filtered"]["diversity"] += 1
            return None

        # 通过所有检查
        self.quality_filter.add_existing([instruction])
        self.diversity_ctrl.add(instruction, "classification" if is_cls else "generation")
        self.stats["passed"] += 1

        return task

    def run(self, verbose: bool = True) -> List[Dict]:
        """运行完整管线直到达到目标数量。"""
        max_attempts = self.target_count * 5
        attempt = 0

        while len(self.generated_tasks) < self.target_count and attempt < max_attempts:
            task = self.generate_one()
            if task is not None:
                self.generated_tasks.append(task)

                if verbose and len(self.generated_tasks) % 20 == 0:
                    success_rate = self.stats['passed'] / self.stats['total_attempts'] * 100
                    print(
                        f"  [{len(self.generated_tasks)}/{self.target_count}] "
                        f"通过率: {success_rate:.1f}%"
                    )
            attempt += 1

        if verbose:
            print(f"\n生成完成！")
            print(f"  目标: {self.target_count}, 实际: {len(self.generated_tasks)}")
            print(f"  总尝试: {self.stats['total_attempts']}")
            print(f"  通过率: {self.stats['passed']/self.stats['total_attempts']*100:.1f}%")
            print(f"  过滤原因: {dict(self.stats['filtered'])}")

        return self.generated_tasks

In [ ]:
# 运行 Self-Instruct 管线
pipeline = SelfInstructPipeline(
    seed_tasks=SEED_TASKS,
    target_count=100,
)

generated_data = pipeline.run(verbose=True)

In [ ]:
# 查看生成结果样例
print("===== 生成样例 =====")
for i, task in enumerate(generated_data[:5]):
    print(f"\n--- Task {i+1} ---")
    print(f"  Instruction: {task['instruction']}")
    print(f"  Input: {task['input'][:80]}..." if task['input'] else "  Input: (none)")
    print(f"  Output: {task['output'][:100]}...")
    print(f"  Is Classification: {task['is_classification']}")

## Part 3：数据质量分析

### 3.1 统计分析

分析生成数据的分布特征，与 Alpaca 52K 的统计特征对比。

In [ ]:
def analyze_dataset(data: List[Dict], name: str = "Dataset"):
    """对生成的数据集进行统计分析。"""
    print(f"\n{'='*60}")
    print(f"  数据集分析: {name}")
    print(f"{'='*60}")

    total = len(data)
    print(f"\n总条数: {total}")

    # 有/无 input 分布
    has_input = sum(1 for d in data if d.get("input", "").strip())
    print(f"有 input: {has_input} ({has_input/total*100:.1f}%)")
    print(f"无 input: {total - has_input} ({(total-has_input)/total*100:.1f}%)")

    # 长度分布
    inst_lens = [len(d["instruction"].split()) for d in data]
    out_lens = [len(d["output"].split()) for d in data]

    print(f"\nInstruction 长度 (words):")
    print(f"  平均: {np.mean(inst_lens):.1f}, 中位数: {np.median(inst_lens):.1f}")
    print(f"  最短: {np.min(inst_lens)}, 最长: {np.max(inst_lens)}")

    print(f"\nOutput 长度 (words):")
    print(f"  平均: {np.mean(out_lens):.1f}, 中位数: {np.median(out_lens):.1f}")
    print(f"  最短: {np.min(out_lens)}, 最长: {np.max(out_lens)}")

    # 动词多样性
    verbs = Counter()
    for d in data:
        words = d["instruction"].strip().split()
        if words:
            verbs[words[0].lower().rstrip('.,!?:;')] += 1

    print(f"\n指令首词多样性:")
    print(f"  唯一首词数: {len(verbs)}")
    print(f"  Top 10: {verbs.most_common(10)}")

    # 分类/非分类分布
    if "is_classification" in data[0]:
        cls_count = sum(1 for d in data if d.get("is_classification"))
        print(f"\n分类任务: {cls_count} ({cls_count/total*100:.1f}%)")
        print(f"非分类任务: {total-cls_count} ({(total-cls_count)/total*100:.1f}%)")

    return {
        "total": total,
        "has_input_ratio": has_input / total,
        "avg_inst_len": np.mean(inst_lens),
        "avg_out_len": np.mean(out_lens),
        "unique_verbs": len(verbs),
    }


our_stats = analyze_dataset(generated_data, "Self-Instruct 生成数据")

In [ ]:
# 与 Alpaca 52K 的参考统计对比
alpaca_reference = {
    "total": 52002,
    "has_input_ratio": 0.40,
    "avg_inst_len": 15,    # tokens（近似 words）
    "avg_out_len": 50,
    "unique_verbs": 300,   # 估计值
}

print("\n===== 与 Alpaca 52K 对比 =====")
print(f"{'指标':<25} {'我们的数据':<15} {'Alpaca 52K':<15}")
print("-" * 55)
print(f"{'总条数':<25} {our_stats['total']:<15} {alpaca_reference['total']:<15}")
print(f"{'有 input 比例':<23} {our_stats['has_input_ratio']:<15.2f} {alpaca_reference['has_input_ratio']:<15.2f}")
print(f"{'平均 instruction 长度':<20} {our_stats['avg_inst_len']:<15.1f} {alpaca_reference['avg_inst_len']:<15}")
print(f"{'平均 output 长度':<22} {our_stats['avg_out_len']:<15.1f} {alpaca_reference['avg_out_len']:<15}")
print(f"{'唯一首词数':<24} {our_stats['unique_verbs']:<15} {alpaca_reference['unique_verbs']:<15}")

### 3.2 质量评估框架

定义一套评估指标来衡量生成数据的质量。

In [ ]:
def evaluate_data_quality(data: List[Dict]) -> Dict:
    """
    多维度评估数据质量。

    评估维度:
    - 格式正确率: instruction/output 非空且格式规范
    - 长度合理率: 长度在合理范围内
    - 去重率: 数据集内部的重复度
    - 多样性: 指令首词和任务类型的多样性
    """
    total = len(data)
    metrics = {}

    # 1. 格式正确率
    format_ok = sum(
        1 for d in data
        if d.get("instruction", "").strip() and d.get("output", "").strip()
    )
    metrics["format_rate"] = format_ok / total

    # 2. 长度合理率 (instruction: 5-200 words, output: 3-500 words)
    length_ok = sum(
        1 for d in data
        if 5 <= len(d["instruction"].split()) <= 200
        and 3 <= len(d["output"].split()) <= 500
    )
    metrics["length_rate"] = length_ok / total

    # 3. 数据集内部去重
    unique_instructions = set()
    duplicates = 0
    for d in data:
        inst_lower = d["instruction"].lower().strip()
        if inst_lower in unique_instructions:
            duplicates += 1
        unique_instructions.add(inst_lower)
    metrics["unique_rate"] = 1 - duplicates / total

    # 4. 动词多样性 (用 unique verbs / total 作为近似)
    verbs = set()
    for d in data:
        words = d["instruction"].strip().split()
        if words:
            verbs.add(words[0].lower())
    metrics["verb_diversity"] = len(verbs) / total

    # 综合评分
    metrics["overall_score"] = (
        metrics["format_rate"] * 0.3
        + metrics["length_rate"] * 0.2
        + metrics["unique_rate"] * 0.3
        + min(metrics["verb_diversity"] * 5, 1.0) * 0.2
    )

    print("===== 数据质量评估 =====")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    return metrics


quality_metrics = evaluate_data_quality(generated_data)

## Part 4：扩展——Evol-Instruct 思路实现

WizardLM 提出的 Evol-Instruct 方法通过渐进式增加指令复杂度来提升数据质量。
这里实现一个简化版本，演示其核心思想。

### 4.1 进化策略

Evol-Instruct 定义了两种进化方向：
- **深度进化**：增加约束、条件、复杂度
- **广度进化**：生成同类但不同形式的指令

In [ ]:
DEPTH_EVOLUTION_PROMPTS = [
    "Add more constraints or requirements to the instruction to make it more complex:\n\n"
    "Original: {instruction}\n\nEvolved:",

    "Make the instruction require multi-step reasoning:\n\n"
    "Original: {instruction}\n\nEvolved:",

    "Add a specific context or scenario that makes the task more challenging:\n\n"
    "Original: {instruction}\n\nEvolved:",
]

BREADTH_EVOLUTION_PROMPTS = [
    "Generate a completely different task that tests similar skills:\n\n"
    "Original: {instruction}\n\nNew task:",

    "Create a variation of this task in a different domain:\n\n"
    "Original: {instruction}\n\nVariation:",
]

# 模拟进化（规则版本）
COMPLEXITY_ADDITIONS = [
    " Ensure the response is under 100 words.",
    " Include at least 3 specific examples.",
    " Consider both advantages and disadvantages.",
    " Provide your reasoning step by step.",
    " Write in a formal academic tone.",
    " Include relevant statistics or data.",
    " Address potential counterarguments.",
    " Organize your response with clear headings.",
]


def evolve_instruction(
    instruction: str,
    evolution_type: str = "depth"
) -> str:
    """模拟指令进化。"""
    if evolution_type == "depth":
        addition = random.choice(COMPLEXITY_ADDITIONS)
        return instruction.rstrip('.') + "." + addition
    else:
        words = instruction.split()
        if len(words) > 3:
            core_task = ' '.join(words[1:])
            new_verbs = ["Analyze", "Evaluate", "Discuss", "Compare",
                         "Investigate", "Assess", "Examine"]
            return random.choice(new_verbs) + " " + core_task
        return instruction


# 演示指令进化
original = "Explain the concept of machine learning."
print(f"原始指令: {original}\n")

print("深度进化:")
for i in range(3):
    evolved = evolve_instruction(original, "depth")
    print(f"  Round {i+1}: {evolved}")

print("\n广度进化:")
for i in range(3):
    evolved = evolve_instruction(original, "breadth")
    print(f"  Variant {i+1}: {evolved}")

In [ ]:
def evolve_dataset(
    seed_data: List[Dict],
    num_rounds: int = 3,
    depth_ratio: float = 0.6
) -> List[Dict]:
    """
    对种子数据集进行多轮进化。

    Args:
        seed_data: 初始数据集
        num_rounds: 进化轮数
        depth_ratio: 深度进化的比例
    """
    all_data = list(seed_data)
    current_pool = list(seed_data)

    for round_idx in range(num_rounds):
        new_tasks = []
        for task in current_pool:
            evo_type = "depth" if random.random() < depth_ratio else "breadth"
            evolved_inst = evolve_instruction(task["instruction"], evo_type)
            evolved_output = simulate_llm_output_generation(evolved_inst)

            new_task = {
                "instruction": evolved_inst,
                "input": task.get("input", ""),
                "output": evolved_output,
                "evolution_round": round_idx + 1,
                "evolution_type": evo_type,
            }
            new_tasks.append(new_task)

        all_data.extend(new_tasks)
        current_pool = new_tasks
        print(f"  Round {round_idx+1}: 生成 {len(new_tasks)} 条, 累计 {len(all_data)} 条")

    return all_data


print("===== Evol-Instruct 进化 =====")
evolved_data = evolve_dataset(SEED_TASKS[:5], num_rounds=3)

print(f"\n进化结果样例:")
for task in evolved_data[-3:]:
    print(f"  [Round {task.get('evolution_round', 0)}, {task.get('evolution_type', 'seed')}]")
    print(f"  Instruction: {task['instruction'][:100]}")
    print()

## Part 5：真实 API 调用版本（可选）

如果你有 OpenAI / Anthropic / 其他 LLM API 的访问权限，可以使用以下代码进行真实的数据生成。

**注意**：实际运行需要设置 API Key 并会产生费用。

In [ ]:
class RealLLMInstructGenerator:
    """
    使用真实 LLM API 的 Self-Instruct 管线。
    支持 OpenAI 兼容接口。

    使用前需安装: pip install openai
    并设置环境变量: OPENAI_API_KEY
    """

    def __init__(self, model: str = "gpt-3.5-turbo", api_key: str = None):
        self.model = model
        self.api_key = api_key
        self._client = None

    @property
    def client(self):
        if self._client is None:
            try:
                import openai
                self._client = openai.OpenAI(api_key=self.api_key)
            except ImportError:
                raise ImportError("请安装 openai: pip install openai")
        return self._client

    def generate_instructions(
        self,
        seed_tasks: List[Dict],
        num_to_generate: int = 20
    ) -> List[str]:
        """
        使用 Alpaca 风格的 prompt 批量生成指令。
        每次调用生成 num_to_generate 条指令。
        """
        seed_examples = random.sample(seed_tasks, min(8, len(seed_tasks)))

        prompt = (
            f"You are asked to come up with a set of {num_to_generate} diverse task instructions.\n"
            "These task instructions will be given to a GPT model.\n\n"
            "Requirements:\n"
            "1. Try not to repeat the verb for each instruction to maximize diversity.\n"
            "2. The language used should be diverse.\n"
            "3. The type of instructions should be diverse (generation, classification, editing, etc.).\n"
            "4. A GPT model should be able to complete the instruction.\n"
            "5. Each instruction should be 1 to 2 sentences long.\n\n"
            "Here are some examples:\n\n"
        )

        for i, task in enumerate(seed_examples, 1):
            prompt += f"{i}. Instruction: {task['instruction']}\n"
            if task['input']:
                prompt += f"   Input: {task['input']}\n"
            prompt += f"   Output: {task['output']}\n\n"

        prompt += f"Now generate {num_to_generate} new and diverse task instructions with inputs and outputs:\n"

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=4096,
            )
            return self._parse_response(response.choices[0].message.content)
        except Exception as e:
            print(f"API 调用失败: {e}")
            return []

    def _parse_response(self, text: str) -> List[Dict]:
        """解析 LLM 返回的文本，提取结构化指令数据。"""
        tasks = []
        current = {}

        for line in text.split("\n"):
            line = line.strip()
            if not line:
                if current.get("instruction"):
                    current.setdefault("input", "")
                    current.setdefault("output", "")
                    tasks.append(current)
                    current = {}
                continue

            for prefix in ["Instruction:", "instruction:"]:
                if prefix in line:
                    current["instruction"] = line.split(prefix, 1)[1].strip()
            for prefix in ["Input:", "input:"]:
                if prefix in line:
                    current["input"] = line.split(prefix, 1)[1].strip()
            for prefix in ["Output:", "output:"]:
                if prefix in line:
                    current["output"] = line.split(prefix, 1)[1].strip()

        if current.get("instruction"):
            current.setdefault("input", "")
            current.setdefault("output", "")
            tasks.append(current)

        return tasks


# 使用示例（需要 API Key，取消注释运行）:
#
# import os
# generator = RealLLMInstructGenerator(
#     model="gpt-3.5-turbo",
#     api_key=os.environ.get("OPENAI_API_KEY")
# )
# real_data = generator.generate_instructions(SEED_TASKS, num_to_generate=20)
# for d in real_data[:3]:
#     print(json.dumps(d, indent=2, ensure_ascii=False))

print("真实 API 版本已定义。如有 API Key，取消注释上方代码运行。")

## Part 6：保存生成数据

将生成的数据保存为 Alpaca 格式的 JSON 文件，供 Day 6 的微调实验使用。

In [ ]:
def save_alpaca_format(data: List[Dict], filepath: str):
    """保存为 Alpaca 标准格式。"""
    alpaca_data = []
    for d in data:
        alpaca_data.append({
            "instruction": d["instruction"],
            "input": d.get("input", ""),
            "output": d["output"],
        })

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(alpaca_data, f, ensure_ascii=False, indent=2)

    print(f"已保存 {len(alpaca_data)} 条数据到 {filepath}")

    # 展示前两条
    print("\n样例:")
    for item in alpaca_data[:2]:
        print(json.dumps(item, ensure_ascii=False, indent=2))


save_alpaca_format(generated_data, "self_instruct_generated.json")

## 总结与自检

### 本日核心收获

```
1. Self-Instruct 四阶段流程:
   指令生成 → 分类判断 → 实例生成 → 质量过滤

2. 关键设计决策:
   - 种子任务的多样性直接决定了生成数据的质量
   - 分类/非分类任务需要不同的实例生成策略
   - ROUGE-L 去重 + 多样性控制是质量保障的关键

3. Alpaca 简化 vs 原始 Self-Instruct:
   - Alpaca 省略了分类判断和多轮迭代
   - 每次生成 20 条（而非 1 条）提高效率
   - 简化牺牲了一些质量，换取了极大的效率提升

4. Evol-Instruct 进化策略:
   - 深度进化增加任务复杂度
   - 广度进化增加任务多样性
   - WizardLM 证明了进化数据的有效性
```

### 自检题

1. **Self-Instruct 的四个阶段分别是什么？** 每个阶段的作用？
2. **为什么要区分分类任务和非分类任务？** 两种生成策略有何不同？
3. **ROUGE-L 在 Self-Instruct 中扮演什么角色？** 阈值设为 0.7 的直觉是什么？
4. **Alpaca 对 Self-Instruct 做了哪些简化？** 这些简化有何利弊？
5. **什么是 Evol-Instruct？** 深度进化和广度进化分别解决什么问题？
6. **如何评估生成数据的质量？** 列举至少 3 个评估维度。

### 产出清单

- [ ] 理解 Self-Instruct 论文的四阶段流程
- [ ] 手写 ROUGE-L 计算函数
- [ ] 实现完整的 Self-Instruct 管线（含质量过滤和多样性控制）
- [ ] 生成至少 100 条指令数据并进行质量分析
- [ ] 了解 Evol-Instruct 的进化策略
- [ ] 保存生成数据为 Alpaca 格式，供 Day 6 使用